# Modbus/TCP PCAP Feature Extraction for Anomaly Detection

This notebook extracts Modbus/TCP fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection.

In [1]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [2]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os
import numpy as np

In [3]:
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """
    Extract Modbus/TCP fields from a PCAP file using tshark.
    
    Args:
        pcap_path: Path to the PCAP file
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract (mbtcp = Modbus/TCP header, modbus = PDU)
    fields = [
        "frame.number",
        "frame.time_epoch",        
        "frame.len",                # Total frame length                   ANOMALOUYS?
        "_ws.col.protocol",         # Protocols in frame
        "ip.src",
        "ip.dst",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",                  # TCP segment length                   ANOMALOUYS?
        "mbtcp.trans_id",           # Transaction Identifier               ANOMALOUYS?
        "mbtcp.prot_id",
        "mbtcp.len",                # Length of remaining bytes in PDU     ANOMALOUYS?
        "mbtcp.unit_id",            # Unit Identifier                      ANOMALOUYS?
        "modbus.func_code",         # Function code                        ANOMALOUYS?
        "modbus.reference_num",     # Reference number (address)           ANOMALOUYS?
        "modbus.word_cnt",          # Number of data words in PDU          ANOMALOUYS?
        "modbus.bit_cnt",           # Number of data bits in PDU
        "modbus.byte_cnt",          # Number of data bytes in PDU          ANOMALOUYS?
        "modbus.exception_code",    # Exception code if present            ANOMALOUYS?
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", pcap_path,  
        "-T", "fields",
        "-Y", "modbus",  # Uncomment to filter only Modbus packets
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]

    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    # Process each line
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'protocols': parse_str(row.get('_ws.col.protocol')),
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'src_port': parse_int(row.get('tcp.srcport')),
            'dst_port': parse_int(row.get('tcp.dstport')),
            'tcp_len': parse_int(row.get('tcp.len')),
            'transaction_id': parse_int(row.get('mbtcp.trans_id')),
            'protocol_id': parse_int(row.get('mbtcp.prot_id')),
            'modbus_length': parse_int(row.get('mbtcp.len')),
            'unit_id': parse_int(row.get('mbtcp.unit_id')),
            'function_code': parse_int(row.get('modbus.func_code')),
            'reference_num': parse_int(row.get('modbus.reference_num')),
            'word_count': parse_int(row.get('modbus.word_cnt')),
            'bit_count': parse_int(row.get('modbus.bit_cnt')),
            'byte_count': parse_int(row.get('modbus.byte_cnt')),
            'exception_code': parse_int(row.get('modbus.exception_code')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # Derive additional features
        record['is_request'] = 1 if record['dst_port'] == 502 else 0
        record['is_response'] = 1 if record['src_port'] == 502 else 0
        record['is_exception'] = 1 if record['exception_code'] is not None else 0
        
        records.append(record)
    
    return records

In [4]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names."""
    df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [5]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    import numpy as np
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Requests per second (rolling 1-second window) - optimized with searchsorted
    timestamps = df['timestamp'].values
    is_request = df['is_request'].values
    
    # Cumulative request count for efficient range queries
    cumulative_requests = np.cumsum(is_request)
    
    # For each packet, count requests in [timestamp - 1, timestamp]
    requests_per_second = np.zeros(len(df), dtype=int)
    for i, ts in enumerate(timestamps):
        # Find index of first packet within the 1-second window
        start_idx = np.searchsorted(timestamps, ts - 1, side='left')
        # Count requests from start_idx to i (inclusive)
        if start_idx > 0:
            requests_per_second[i] = cumulative_requests[i] - cumulative_requests[start_idx - 1]
        else:
            requests_per_second[i] = cumulative_requests[i]
    
    df['requests_per_second'] = requests_per_second
    
    return df

In [6]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df


def add_rtt_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add round trip time (RTT) by matching Modbus requests with responses.
    
    RTT is calculated as the time between a request and its corresponding response,
    matched by transaction_id and IP address pairs.
    """
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['round_trip_time'] = np.nan
    
    # Dictionary to track pending requests: (transaction_id, client_ip, server_ip) -> (index, timestamp)
    pending_requests = {}
    
    for idx, row in df.iterrows():
        trans_id = row['transaction_id']
        
        if row['is_request'] == 1:
            # This is a request (going to port 502)
            # Key: (transaction_id, client_ip, server_ip)
            key = (trans_id, row['src_ip'], row['dst_ip'])
            pending_requests[key] = (idx, row['timestamp'])
            
        elif row['is_response'] == 1:
            # This is a response (coming from port 502)
            # Key: (transaction_id, client_ip, server_ip) - note src/dst are swapped
            key = (trans_id, row['dst_ip'], row['src_ip'])
            
            if key in pending_requests:
                req_idx, req_timestamp = pending_requests[key]
                rtt = row['timestamp'] - req_timestamp
                
                # Set RTT on both the response and the original request
                df.at[idx, 'round_trip_time'] = rtt
                df.at[req_idx, 'round_trip_time'] = rtt
                
                # Remove the matched request
                del pending_requests[key]
    
    return df

def add_flooding_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add features specific to query flooding detection."""
    
    # Sort by timestamp
    df = df.sort_values('timestamp')
    
    # Transaction ID patterns
    df['transaction_id_delta'] = df.groupby('flow_id')['transaction_id'].diff().fillna(0)
    df['transaction_id_repeats'] = df.groupby('flow_id')['transaction_id'].transform(
        lambda x: x.duplicated(keep=False).sum()
    )
    
    # Function code analysis
    df['func_code_entropy'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: -sum(p * np.log2(p) for p in x.value_counts(normalize=True) if p > 0)
    )
    df['consecutive_same_func_code'] = (
        df.groupby('flow_id')['function_code']
        .transform(lambda x: (x == x.shift()).cumsum())
    )
    
    # Request/Response ratios per flow
    df['flow_request_count'] = df.groupby('flow_id')['is_request'].transform('sum')
    df['flow_response_count'] = df.groupby('flow_id')['is_response'].transform('sum')
    df['response_rate'] = df['flow_response_count'] / df['flow_request_count'].replace(0, 1)
    
    # Exception rate
    df['flow_exception_count'] = df.groupby('flow_id')['is_exception'].transform('sum')
    df['exception_rate'] = df['flow_exception_count'] / df['flow_pkt_count'].replace(0, 1)
    
    # Rolling window features (for time-based flooding patterns)
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.set_index('timestamp_dt')
    
    df = df.reset_index(drop=True)
    
    return df

## Usage Example

In [7]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
relative_path = "data\\raw\\captures1_v2\\mitm\\eth2dump-mitm-change-15m-1h_1.pcap"
absolute_path = (Path("../..") / relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = absolute_path  # Update this path
OUTPUT_CSV = "modbus_features.csv"

Using PCAP file at: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-15m-1h_1.pcap


In [8]:
# Extract raw Modbus features
print(f"Processing: {PCAP_PATH}")
records = extract_modbus_features(PCAP_PATH)
print(f"Extracted {len(records)} Modbus packets")

Processing: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-15m-1h_1.pcap
Extracted 24234 Modbus packets


In [9]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = add_temporal_features(df)
    df = add_flow_features(df)
    df = add_rtt_feature(df)
    df = add_flooding_features(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")


DataFrame shape: (24234, 40)

Columns: ['packet_number', 'timestamp', 'datetime', 'protocols', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'tcp_len', 'transaction_id', 'protocol_id', 'modbus_length', 'unit_id', 'function_code', 'reference_num', 'word_count', 'bit_count', 'byte_count', 'exception_code', 'pkt_len', 'is_request', 'is_response', 'is_exception', 'function_name', 'inter_arrival_time', 'time_from_start', 'requests_per_second', 'flow_id', 'flow_pkt_count', 'unique_func_codes', 'round_trip_time', 'transaction_id_delta', 'transaction_id_repeats', 'func_code_entropy', 'consecutive_same_func_code', 'flow_request_count', 'flow_response_count', 'response_rate', 'flow_exception_count', 'exception_rate']


In [10]:
# Preview the data
df.head(10)

,packet_number,timestamp,datetime,protocols,src_ip,dst_ip,src_port,dst_port,tcp_len,transaction_id,...,round_trip_time,transaction_id_delta,transaction_id_repeats,func_code_entropy,consecutive_same_func_code,flow_request_count,flow_response_count,response_rate,flow_exception_count,exception_rate
0,3,1.535070e+09,2018-08-23T20:16:22.479840,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,0.011384,0.0,22993,-0.0,0,11484,11509,1.002177,0,0.0
1,4,1.535070e+09,2018-08-23T20:16:22.491224,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,0.011384,0.0,22993,-0.0,1,11484,11509,1.002177,0,0.0
2,6,1.535070e+09,2018-08-23T20:16:22.791817,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,0.010061,0.0,22993,-0.0,2,11484,11509,1.002177,0,0.0
3,7,1.535070e+09,2018-08-23T20:16:22.801878,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,0.010061,0.0,22993,-0.0,3,11484,11509,1.002177,0,0.0
4,9,1.535070e+09,2018-08-23T20:16:23.103806,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,0.007421,0.0,22993,-0.0,4,11484,11509,1.002177,0,0.0
5,10,1.535070e+09,2018-08-23T20:16:23.111227,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,0.007421,0.0,22993,-0.0,5,11484,11509,1.002177,0,0.0
6,18,1.535070e+09,2018-08-23T20:16:23.416902,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,0.004302,0.0,22993,-0.0,6,11484,11509,1.002177,0,0.0
7,19,1.535070e+09,2018-08-23T20:16:23.421204,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,0.004302,0.0,22993,-0.0,7,11484,11509,1.002177,0,0.0
8,21,1.535070e+09,2018-08-23T20:16:23.727798,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,0.004078,0.0,22993,-0.0,8,11484,11509,1.002177,0,0.0
9,22,1.535070e+09,2018-08-23T20:16:23.731876,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,0.004078,0.0,22993,-0.0,9,11484,11509,1.002177,0,0.0


In [11]:
# Summary statistics
print("Function Code Distribution:")
print(df['function_name'].value_counts())
print("\nBasic Statistics:")
df[['pkt_len', 'inter_arrival_time', 'modbus_length']].describe()

Function Code Distribution:
function_name
Read Holding Registers    22993
Write Single Register      1241
Name: count, dtype: int64

Basic Statistics:


,pkt_len,inter_arrival_time,modbus_length
count,24234.000000,24233.000000,24234.000000
mean,75.034703,0.148525,15.023314
std,9.563830,0.147938,9.488229
min,66.000000,0.000271,6.000000
25%,66.000000,0.007250,6.000000
50%,66.000000,0.072318,6.000000
75%,85.000000,0.304097,25.000000
max,234.000000,2.900035,25.000000


In [12]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")


Saved to: modbus_features.csv


## Feature Summary for Anomaly Detection

| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Protocol** | `function_code`, `unit_id`, `transaction_id` | Detect unauthorized commands |
| **Payload** | `reference_num`, `word_count`, `byte_count`, `reg_value` | Detect data manipulation |
| **Temporal** | `inter_arrival_time`, `rolling_iat_*`, `rtt` | Detect timing anomalies, DoS, network issues |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |
| **Error** | `is_exception`, `exception_code` | Detect probing, fuzzing |